# 第 3 周练习 —— 传感器合成数据集生成器

## 练习目标（理念）

在 **Google Colab** 上搭一个小工具：用本地加载的开源因果语言模型（Causal LM）生成**合成传感器表格数据**（CSV 风格），方便后续做 IoT / 时序实验。

- **输入**：传感器类型（温度 / 湿度 / 转速计）、行数、可选模型、是否 4-bit 量化
- **输出**：仅表格文本（timestamp、sensor_id、读数等列），可从 Gradio 界面复制或下载使用
- **交互**：用 **Gradio** `Blocks` 做配置面板与一键生成

这是第 3 周常见主题的延伸：Hugging Face `transformers` 加载 Instruct 模型、BitsAndBytes 量化省显存、流式 `generate` + Gradio UI。

## 和本课第 3 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Hugging Face 模型加载 | `AutoTokenizer` / `AutoModelForCausalLM.from_pretrained` |
| 4-bit 量化（BitsAndBytes） | `BitsAndBytesConfig(load_in_4bit=True, …)` |
| Chat 模板 | `tokenizer.apply_chat_template(...)` |
| 流式生成 | `TextIteratorStreamer` + 后台 `Thread` |
| Gradio 界面 | `gr.Blocks`、Dropdown / Checkbox / Button |

## 怎么跑

1. 建议在 **Google Colab**（带 GPU）从上到下运行；本地需自行安装同名依赖与 CUDA
2. 在 Colab Secrets 配置 `HF_TOKEN`（Hugging Face 访问令牌），部分模型需同意许可
3. 运行最后一格启动 Gradio：选模型 → 勾选是否 4-bit → 选传感器类型与行数 → Generate
4. Prompt / model id / UI 英文文案保持原样（影响模型行为与界面），教学说明用中文注释

## 功能要点

- 从多种 Colab 友好的 Instruct 模型中选择，生成表格数据
- 支持 **4-bit 量化**：更高效率、更低显存（VRAM）占用
- Gradio 接口：配置、生成、查看数据集


In [ ]:
# ========== 依赖安装：Colab 环境补齐量化与 transformers 版本 ==========
# -q：安静模式少刷屏；--upgrade：尽量升到可用版本
# bitsandbytes：4-bit / 8-bit 量化后端；accelerate：device_map="auto" 等加速加载
# transformers==4.53.3：锁定版本，降低与 BitsAndBytes / 模型代码的兼容性风险
!pip install -q --upgrade bitsandbytes accelerate "transformers==4.53.3"


In [ ]:
# ========== 导入：标准库 + Colab + Hugging Face + Gradio ==========

# os：环境变量、路径等（本格未直接用到，常与密钥/配置一起导入）
import os
# re：正则（预留；清洗模型输出时常用）
import re
# sys：检测是否在 google.colab 模块里，决定 Gradio 是否 share
import sys
# Thread：把 model.generate 放到后台线程，主线程消费 streamer
from threading import Thread
# requests：HTTP 客户端（本笔记本主路径用 transformers 本地推理，库仍导入）
import requests
# IPython 展示工具：Markdown / 动态更新显示（本练习主 UI 走 Gradio）
from IPython.display import Markdown, display, update_display
# OpenAI 客户端：课程里常用；本练习生成路径以本地 HF 模型为主
from openai import OpenAI
# Colab：drive 可挂载云盘；userdata 读 Secrets（如 HF_TOKEN）
from google.colab import drive, userdata
# transformers：分词器、因果 LM、量化配置、流式文本输出
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TextIteratorStreamer,
    TextStreamer,
)
# huggingface_hub.login：用 token 登录，以便拉取需授权的模型
from huggingface_hub import login
# torch：张量与 dtype（如 bfloat16）、CUDA 缓存清理
import torch
# gradio：快速搭 Web UI（Blocks / Dropdown / Button 等）
import gradio as gr

# 从 Colab Secrets 读取 Hugging Face Token（不要把密钥写进笔记本正文）
hf_token = userdata.get('HF_TOKEN')
# 登录 Hub；add_to_git_credential=True 便于后续 git/lfs 凭据复用
login(hf_token, add_to_git_credential=True)


In [ ]:
# ========== 配置：可选模型、传感器类型、英文 Prompt 模板 ==========

# 展示名 → Hugging Face model id；下拉框用 key，加载用 value（勿改 model id 字符串）
MODELS = {
    "Llama 3.2 3B": "meta-llama/Llama-3.2-3B-Instruct",
    "Phi-4 mini": "microsoft/Phi-4-mini-instruct",
    "Gemma 3 270M": "google/gemma-3-270m-it",
    "Qwen3 4B": "Qwen/Qwen3-4B-Instruct-2507",
    "DeepSeek R1 Distill 1.5B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
}

# 三种传感器类型：与 PROMPTS 的 key、Gradio Dropdown choices 一致
SENSOR_TYPES = ["temperature", "humidity", "tachometer"]

# 每个传感器一条英文 user prompt；{n} 占位行数（.format(n=...) 注入）
# 影响模型行为的字符串保持英文，不要翻译
PROMPTS = {
    "temperature": "Generate a CSV dataset for a temperature sensor. Columns: timestamp, sensor_id, temperature_celsius, location, unit. Include {n} rows of realistic readings (e.g. room, outdoor, machine). Return only the table, no explanation.",
    "humidity": "Generate a CSV dataset for a humidity sensor. Columns: timestamp, sensor_id, humidity_percent, location, unit. Include {n} rows. Return only the table.",
    "tachometer": "Generate a CSV dataset for a tachometer (RPM). Columns: timestamp, sensor_id, rpm, machine_id, unit. Include {n} rows of realistic rotation readings. Return only the table.",
}


In [ ]:
# ========== 模型加载：4-bit 量化配置 + from_pretrained ==========

def get_quantization_config():
    # 返回 BitsAndBytes 4-bit 配置：省 VRAM，Colab T4 类 GPU 更友好
    return BitsAndBytesConfig(
        # 启用 4-bit 权重量化加载
        load_in_4bit=True,
        # 双重量化：再压缩量化常数，进一步省显存
        bnb_4bit_use_double_quant=True,
        # 计算时用 bfloat16，平衡速度与数值稳定
        bnb_4bit_compute_dtype=torch.bfloat16,
        # 量化类型 nf4：NormalFloat4，常用高质量 4-bit 方案
        bnb_4bit_quant_type="nf4",
    )


def load_model(model_id: str, use_quantization: bool):
    # 按 model_id 拉分词器；trust_remote_code=True 允许仓库自定义代码（部分模型需要）
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    # 若没有 pad_token，用 eos_token 顶上，避免 generate 时 pad 报错
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if use_quantization:
        # 走 4-bit：挂上 quantization_config，device_map="auto" 自动分配设备
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=get_quantization_config(),
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        # 不量化：整模 bfloat16 + auto device_map（显存需求更高）
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
        )
    # 返回分词器与模型，供后续聊天模板与 generate 使用
    return tokenizer, model


In [ ]:
# ========== 缓存：同一 (model_id, 量化开关) 复用，避免重复加载 ==========

# 模块级缓存：当前已加载的模型 id / 是否量化 / tokenizer / model
_current_model_id = None
_use_quantization = None
_tokenizer = None
_model = None


def get_or_load_model(model_id: str, use_quantization: bool):
    """仅在选择更改时加载模型；否则重复使用。"""
    # 声明改的是模块全局缓存，而不是函数局部变量
    global _current_model_id, _use_quantization, _tokenizer, _model
    # 命中缓存：id 与量化开关都相同 → 直接返回，省下载与显存抖动
    if _tokenizer is not None and _model is not None and _current_model_id == model_id and _use_quantization == use_quantization:
        return _tokenizer, _model
    # 已有旧模型：先删引用，再 empty_cache，尽量释放 GPU 显存
    if _model is not None:
        del _model
        if _tokenizer is not None:
            del _tokenizer
        torch.cuda.empty_cache()
    # 更新缓存元数据，再真正 load
    _current_model_id = model_id
    _use_quantization = use_quantization
    _tokenizer, _model = load_model(model_id, use_quantization)
    return _tokenizer, _model


In [ ]:
# ========== 生成：拼 messages → chat template → 流式 generate ==========

def build_messages(sensor_type: str, n_rows: int):
    # system：约束「只输出表格」；英文 prompt 保持原样（影响模型行为）
    system = "You are a dataset engineer. Generate only the requested sensor data table. Output ONLY the table, no extra text, code blocks, or explanations."
    # 按传感器类型取模板，缺省退回 temperature；用 n_rows 填 {n}
    user_text = PROMPTS.get(sensor_type, PROMPTS["temperature"]).format(n=n_rows)
    # OpenAI 风格 messages：system + user，交给 apply_chat_template
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user_text},
    ]

def generate_dataset(model_name: str, use_quantization: bool, sensor_type: str, n_rows: int):
    # 展示名 → Hub model id；未知名时退回字典里第一个 id
    model_id = MODELS.get(model_name, list(MODELS.values())[0])
    # 带缓存地加载（或复用）tokenizer 与 model
    tokenizer, model = get_or_load_model(model_id, use_quantization)
    # 构造本轮对话 messages
    messages = build_messages(sensor_type, n_rows)

    # 输入张量应放到与模型参数相同的 device（accelerate 分片时取第一个参数所在设备）
    device = next(model.parameters()).device if hasattr(model, "parameters") else "cuda"
    try:
        # 优先用模型自带 chat template，拼成带 generation prompt 的字符串
        text = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        # 再 tokenize 成张量并搬到 device
        inputs = tokenizer(text, return_tensors="pt").to(device)
    except Exception:
        # 无 chat template 或不兼容时：退化为只拼 user 文本
        prompt = "\n".join(m.get("content", "") for m in messages if m.get("role") == "user")
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # TextIteratorStreamer：generate 时边生成边 yield 解码后的文本块
    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, skip_special_tokens=True
    )
    # 在后台线程跑 generate，避免阻塞主线程读 streamer
    thread = Thread(
        target=lambda: model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            # 最多新生成 4096 token，表格较长时留余量
            max_new_tokens=4096,
            # 采样而非贪心，温度 0.7 增加多样性
            do_sample=True,
            temperature=0.7,
            # pad_token_id 必填，避免警告/异常；没有 pad 就用 eos
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            streamer=streamer,
        )
    )
    thread.start()
    # 主线程拼接 streamer 吐出的所有 chunk
    out = ""
    for chunk in streamer:
        out += chunk

    # 去掉首尾空白与可能包住的 markdown 代码围栏 ```
    out = out.strip().removeprefix("```").removesuffix("```").strip()
    return out.strip()


In [ ]:
# ========== Gradio UI：参数面板 + 一键生成 + launch ==========

def run_ui(model_name: str, use_quantization: bool, sensor_type: str, n_rows: int):
    # 把行数夹在 [5, 500]；非法输入则默认 50
    try:
        n = max(5, min(int(n_rows), 500))
    except (TypeError, ValueError):
        n = 50
    # 调用上一格的生成逻辑，返回表格文本给 Textbox
    return generate_dataset(model_name, use_quantization, sensor_type, n)

# Blocks：组合式布局；title / Soft 主题字符串保持原样（界面文案不翻译）
with gr.Blocks(title="Sensor Dataset Generator", theme=gr.themes.Soft()) as demo:
    # 标题与说明（Markdown 字符串影响 UI 显示，保持英文）
    gr.Markdown("## Sensor Dataset Generator")
    gr.Markdown("Generate synthetic **temperature**, **humidity**, or **tachometer** datasets. Choose model and quantization to compare outputs.")

    # 第一行：模型下拉 + 是否 4-bit 量化
    with gr.Row():
        model_dd = gr.Dropdown(
            choices=list(MODELS.keys()),
            value=list(MODELS.keys())[0],
            label="Model",
        )
        quant_cb = gr.Checkbox(value=True, label="Use 4-bit quantization (saves VRAM)")
    # 第二行：传感器类型 + 行数
    with gr.Row():
        sensor_dd = gr.Dropdown(
            choices=SENSOR_TYPES,
            value=SENSOR_TYPES[0],
            label="Sensor type",
        )
        n_rows_num = gr.Number(value=50, label="Number of rows", minimum=5, maximum=500, step=5)

    # 生成按钮与输出文本框
    gen_btn = gr.Button("Generate dataset")
    out_text = gr.Textbox(label="Generated dataset", lines=20, max_lines=30)

    # 点击按钮：把四个控件值传给 run_ui，结果写入 out_text
    gen_btn.click(fn=run_ui, inputs=[model_dd, quant_cb, sensor_dd, n_rows_num], outputs=out_text)

    # Colab 里常需 share=True 才能从外网打开公网链接；本地则 False
    in_colab = "google.colab" in sys.modules
    demo.launch(share=in_colab, show_error=True)
